# 15 · Infix, Postfix & Prefix Expressions

The same expression can be written three ways; they only differ in **where the operator goes**:

| Notation | Operator position | `(x + y) × (a − b)` |
|---|---|---|
| **Infix** | between operands | `(x+y)*(a-b)` |
| **Prefix** | **before** its operands | `*+xy-ab` |
| **Postfix** | **after** its operands | `xy+ab-*` |

Infix needs parentheses and precedence rules. Prefix and postfix need **neither**, which makes them easy for computers to evaluate.

**Precedence (high → low):** `^` (3), `*` `/` (2), `+` `-` (1). `^` is **right-associative** (`a^b^c = a^(b^c)`); the others are left-associative.

The code is exactly my original; I added explanations/comments. *(Those cells only define classes, so there is no printed output. Example results are listed in the text.)*

**Topics:** infix → postfix · infix → prefix · postfix → infix · prefix → infix · postfix → prefix · prefix → postfix

## My original notes

operator :  + , - , * , ** , /


---

Operands : A to Z;
a to z ;
0 to 9 ;


---

* prefix : *+ Pq - mn
* Infix  : (x+y) * (a-b)
* Postfix : pq + mn -*


---

INfix to Postfix



## 1. Infix → Postfix
Scan left to right with an **operator stack** and an **output list**:

| Symbol | Action |
|---|---|
| operand | append to the output |
| `(` | push on the stack |
| `)` | pop operators to the output until `(`, then discard `(` |
| operator | pop operators with **higher or equal precedence** from the stack, then push this one |

At the end, pop everything left on the stack.

Example `a+b*c-d` → **`abc*+d-`** (`*` stays above `+` on the stack so it is output first; when `-` arrives, both `*` and `+` are popped).

> ⚠️ **Note:** popping on `>=` makes **`^` left-associative** too, so `a^b^c` becomes `ab^c^` instead of `abc^^`. For `+ - * /` the code is correct.

In [ ]:
class Solution:
    # higher number = binds tighter
    def precedence(self, ch):
        # Define operator precedence hierarchy
        if ch == "+" or ch == "-":
            return 1
        if ch == "*" or ch == "/":
            return 2
        if ch == "^":
            return 3
        return 0  # For non-operators like '(' and ')'

    # convert an infix string (single-character operands) to postfix
    def InfixtoPostfix(self, s):
        stack = []    # Stack to help with operator ordering
        result = []   # To store the postfix expression

        for char in s:
            # If character is an operand, add it directly to result
            # operand: letter or digit goes straight to the output
            if ("a" <= char <= "z") or ("A" <= char <= "Z") or ("0" <= char <= "9"):
                result.append(char)

            # If character is '(', push it to the stack
            elif char == "(":
                stack.append(char)

            # If character is ')', pop until '(' is encountered
            elif char == ")":
                while stack and stack[-1] != "(":
                    result.append(stack.pop())
                stack.pop()  # Discard the '('

            # If character is an operator
            else:
                # Pop operators with higher or equal precedence
                # pop operators that must run before this one (higher OR equal precedence)
                while stack and self.precedence(stack[-1]) >= self.precedence(char):
                    result.append(stack.pop())
                stack.append(char)  # Push current operator

        # Pop remaining operators from the stack
        while stack:
            result.append(stack.pop())

        return "".join(result)  # Convert list to string

## 2. Infix → Prefix
A prefix expression is the mirror image of a postfix one, so:
1. **Reverse** the infix string
2. **Swap** every `(` with `)`
3. Run the infix → postfix algorithm with the opposite tie-break: pop only when the stack top has **strictly higher** precedence (`>`)
4. **Reverse** the result

Example `(x+y)*(a-b)` → **`*+xy-ab`**; `a+b+c` → **`++abc`**.

> ⚠️ **Note:** the strict `>` is right for left-associative operators, but `^` would need `>=` in this direction, so `a^b^c` is not converted correctly.

In [ ]:
class Solution:
    def precedence(self, ch):
        # Define operator precedence hierarchy
        if ch == "+" or ch == "-":
            return 1
        if ch == "*" or ch == "/":
            return 2
        if ch == "^":
            return 3
        return 0  # For non-operators like '(' and ')'

    def infixToPrefix(self, s):
        # Step 1: Reverse the infix expression
        # step 1: reverse the infix expression
        s = s[::-1]

        # Step 2: Swap opening and closing parentheses
        # step 2: swap "(" and ")" (via a temporary word so they don't overwrite each other)
        s = s.replace("(", "temp").replace(")", "(").replace("temp", ")")

        stack = []
        result = []

        # Step 3: Modified infix to postfix algorithm
        for char in s:
            # If character is an operand, add it directly to result
            if ("a" <= char <= "z") or ("A" <= char <= "Z") or ("0" <= char <= "9"):
                result.append(char)

            # If character is '(', push it to the stack
            elif char == "(":
                stack.append(char)

            # If character is ')', pop until '(' is encountered
            elif char == ")":
                while stack and stack[-1] != "(":
                    result.append(stack.pop())
                stack.pop()  # Discard the '('

            # If character is an operator
            else:
                # Note the change to '>' from '>=' in the infix to postfix algorithm
                # This handles right-associativity properly
                # step 3: strict ">" (opposite tie-break)
                while stack and self.precedence(stack[-1]) > self.precedence(char):
                    result.append(stack.pop())
                stack.append(char)  # Push current operator

        # Pop remaining operators from the stack
        while stack:
            result.append(stack.pop())

        # Step 4: Reverse the result to get prefix expression
        # step 4: reverse the result
        return "".join(result[::-1])

## 3. Postfix → Infix
Use a **stack of sub-expressions** scanning left to right: operand → push; operator → pop two operands (the first one popped is the **second** operand), push `(operand1 operator operand2)`.
Example `xy+ab-*` → `((x+y)*(a-b))`. The result is fully parenthesised.

In [ ]:
class Solution:
    def postToInfix(self, s):
        # Stack to store operands or partial infix expressions
        stack = []

        for char in s:
            # If character is an operand (letter or digit), push it to the stack
            if char.isalnum():
                stack.append(char)
            else:
                # If character is an operator, pop two operands
                # the last operand is on top
                operand2 = stack.pop()  # Second operand (popped first)
                # then the first operand
                operand1 = stack.pop()  # First operand (popped second)

                # Combine operands with the operator, surrounded by parentheses
                # wrap in brackets so the order is unambiguous
                new_expr = f"({operand1}{char}{operand2})"

                # Push the resulting expression back onto the stack
                stack.append(new_expr)

        # The final element in the stack is the complete infix expression
        return stack[-1]

## 4. Prefix → Infix
Same idea, but scan **right to left**: now the **first** popped operand is the **first** operand.
Example `*+xy-ab` → `((x+y)*(a-b))`.

In [ ]:
class Solution:
    def preToInfix(self, s):
        # Stack to store operands or partial infix expressions
        stack = []

        # Process the prefix expression from right to left
        # scan the prefix expression from right to left
        for char in s[::-1]:
            # If character is an operand (letter or digit), push it to the stack
            if char.isalnum():
                stack.append(char)
            else:
                # If character is an operator, pop two operands
                # first operand comes off first in this direction
                operand1 = stack.pop()  # First operand
                operand2 = stack.pop()  # Second operand

                # Combine operands with the operator, surrounded by parentheses
                new_expr = f"({operand1}{char}{operand2})"

                # Push the resulting expression back onto the stack
                stack.append(new_expr)

        # The final element in the stack is the complete infix expression
        return stack[-1]

## 5. Postfix → Prefix
Scan left to right with a stack of **prefix** sub-expressions; for an operator pop `operand2`, then `operand1`, and push `operator + operand1 + operand2`.
Example `xy+ab-*` → `*+xy-ab`.

In [ ]:
class Solution:
    def postToPre(self, s):
        # Stack to store operands or partial prefix expressions
        stack = []

        # Process each character in postfix expression
        for char in s:
            # If the character is an operand, push it to the stack
            if char.isalnum():
                stack.append(char)
            else:
                # Pop two operands from the stack
                operand2 = stack.pop()  # Second operand (popped first)
                operand1 = stack.pop()  # First operand (popped second)

                # Combine the operands with the operator in prefix form
                # operator goes in front
                new_expr = f"{char}{operand1}{operand2}"

                # Push the result back onto the stack
                stack.append(new_expr)

        # The final element in the stack is the prefix expression
        return stack[-1]

## 6. Prefix → Postfix
Scan **right to left** with a stack of **postfix** sub-expressions; for an operator pop `operand1`, then `operand2`, and push `operand1 + operand2 + operator`.
Example `*+xy-ab` → `xy+ab-*`.

In [ ]:
class Solution:
    def preToPost(self, s):
        # Stack to store operands or partial postfix expressions
        stack = []

        # Traverse the prefix expression from right to left using index
        n = len(s)
        # right to left using indexes
        for i in range(n - 1, -1, -1):  # Reverse iteration using index
            char = s[i]

            # If the character is an operand, push it to the stack
            if char.isalnum():
                stack.append(char)
            else:
                # Pop two operands from the stack
                operand1 = stack.pop()  # First operand
                operand2 = stack.pop()  # Second operand

                # Combine the operands with the operator in postfix form
                # operator goes at the back
                new_expr = operand1 + operand2 + char

                # Push the result back onto the stack
                stack.append(new_expr)

        # The final element in the stack is the postfix expression
        return stack[-1]

---
### ✅ Key takeaways
- **Postfix/prefix need no parentheses**; the order of the symbols already fixes the evaluation order.
- Infix → postfix: operand → output, `(` push, `)` pop to `(`, operator → pop higher-or-equal precedence first.
- Infix → prefix = reverse, swap brackets, postfix algorithm with strict `>`, reverse again.
- Postfix and prefix conversions differ only in **scan direction** and **pop order**.